# Alucard v2 — Colab retraining test

This notebook clones `Apache0ne/alucard` branch `fix/training-pipeline-v2`, downloads the published Alucard sprite dataset, precomputes matching CLIP embeddings, trains a controlled test run, and produces fixed-seed RGBA/alpha benchmarks.

The default is a **20k-row test run**. Set `MAX_SAMPLES = 0` for the full dataset after the test run behaves correctly.


In [ ]:
# Experiment settings
BRANCH = "fix/training-pipeline-v2"
MAX_SAMPLES = 20_000      # 0 = all ~313k rows
EPOCHS = 10
BATCH_SIZE = 32
GRAD_ACCUM = 1
PRECISION = "bf16"        # L4/A100/H100: bf16; T4: use "fp16"
NUM_WORKERS = 4
CFG_TEXT_BENCH = 2.5
BENCH_STEPS = 30
SEED = 42
USE_DRIVE = True

DATA_DIR = "/content/alucard_data"
LOCAL_OUT = "/content/alucard_runs/test_v2"
print({k:v for k,v in globals().copy().items() if k in ["BRANCH","MAX_SAMPLES","EPOCHS","BATCH_SIZE","GRAD_ACCUM","PRECISION"]})


In [ ]:
# GPU/runtime information
import os, platform, subprocess, torch
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
    print("BF16 supported:", torch.cuda.is_bf16_supported())


In [ ]:
# Clone the branch and install it
import os, subprocess, pathlib, shutil
os.chdir("/content")
if pathlib.Path("/content/alucard").exists():
    shutil.rmtree("/content/alucard")
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", "https://github.com/Apache0ne/alucard.git"], check=True)
os.chdir("/content/alucard")
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", "."], check=True)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
# Optional: persist checkpoints/benchmarks to Google Drive
from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/AlucardV2/test_v2"
else:
    OUTPUT_DIR = LOCAL_OUT
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR =", OUTPUT_DIR)


In [ ]:
# Download the 128x128 RGBA dataset and precompute ViT-B/32 text embeddings.
# Re-running this cell reuses prepared files unless --force is added.
import subprocess
cmd = [
    "python", "scripts/download_alucard_dataset.py",
    "--output-dir", DATA_DIR,
    "--max-samples", str(MAX_SAMPLES),
    "--embedding-batch-size", "512",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Dataset audit: verify resolution and native alpha statistics on a deterministic sample.
from datasets import load_from_disk
import numpy as np, random, json

ds = load_from_disk(DATA_DIR + "/hf_dataset")
rng = random.Random(123)
indices = rng.sample(range(len(ds)), min(512, len(ds)))
partial = []
strict_partial = []
sizes = set()
for idx in indices:
    img = ds[idx]["image"].convert("RGBA")
    sizes.add(img.size)
    a = np.asarray(img, dtype=np.uint8)[...,3]
    partial.append(((a > 13) & (a < 242)).mean())
    strict_partial.append(((a > 0) & (a < 255)).mean())
print("rows:", len(ds))
print("sampled image sizes:", sorted(sizes))
print("mean target partial alpha 5-95%:", float(np.mean(partial)))
print("mean target strict partial alpha:", float(np.mean(strict_partial)))
print("first captions:")
for i in range(min(8, len(ds))): print(" ", ds[i]["text"])


In [ ]:
# Train the corrected v2 pipeline.
import subprocess
cmd = [
    "alucard-train",
    "--data-dir", DATA_DIR,
    "--output-dir", OUTPUT_DIR,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--precision", PRECISION,
    "--num-workers", str(NUM_WORKERS),
    "--sample-every", "1",
    "--save-every", "1",
    "--val-max-batches", "32",
    "--seed", str(SEED),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Benchmark best.pt with identical starting noise across prompts.
import subprocess, os
BENCH_DIR = OUTPUT_DIR + "/benchmark"
cmd = [
    "python", "scripts/benchmark_checkpoint.py",
    "--checkpoint", OUTPUT_DIR + "/best.pt",
    "--output-dir", BENCH_DIR,
    "--steps", str(BENCH_STEPS),
    "--cfg-text", str(CFG_TEXT_BENCH),
    "--seed", str(SEED),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Display the visual regression sheet and benchmark metrics.
from IPython.display import display
from PIL import Image
import json, pathlib

display(Image.open(BENCH_DIR + "/contact_sheet.png"))
report = json.loads(pathlib.Path(BENCH_DIR + "/benchmark.json").read_text())
print(json.dumps({
    "images_per_second": report["images_per_second"],
    "peak_gpu_memory_gb": report["peak_gpu_memory_gb"],
    "clip_cosine_mean": report["clip_cosine_mean"],
    "alpha_mean": report["alpha_mean"],
}, indent=2))


In [ ]:
# Training curve summary
import json, pathlib
records = [json.loads(x) for x in pathlib.Path(OUTPUT_DIR + "/metrics.jsonl").read_text().splitlines() if x.strip()]
for r in records:
    print(
        f"epoch={r['epoch']:>3} "
        f"train={r['train_total']:.4f} "
        f"val={r['val_total']:.4f} "
        f"val_alpha={r['val_alpha_recon']:.4f} "
        f"preview_partial={r.get('preview_partial_alpha', float('nan')):.4f} "
        f"preview_binary={r.get('preview_binary_alpha', float('nan')):.4f}"
    )


## After the test run

Do not jump to a full 313k-row run unless the contact sheet is visibly moving toward crisp subjects and the partial-alpha metric is falling. For a full preparation, set `MAX_SAMPLES = 0`, use a new `OUTPUT_DIR`, rerun the dataset cell, then train with a larger epoch budget.
